### __Load Environment Variables__

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### __State__

In [2]:
from typing import Annotated,Sequence, TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    """The state of the agent."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    number_of_steps: int

### __Tools__

In [3]:
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
import math

@tool
def triple(num: float) -> float:
    """
    Nhân một số với 3.
    :param num: số cần được nhân ba
    :return: kết quả là số đó nhân với 3
    """
    return 3 * float(num)

@tool
def modulo(num: float, divisor: float) -> float:
    """
    Chia lấy phần dư (modulo operation).
    :param num: số bị chia
    :param divisor: số chia
    :return: phần dư của phép chia num cho divisor
    """
    return float(num) % float(divisor)

@tool
def power(base: float, exponent: float) -> float:
    """
    Tính lũy thừa của một số.
    :param base: cơ số
    :param exponent: số mũ
    :return: base mũ exponent
    """
    return float(base) ** float(exponent)

@tool
def square_root(num: float) -> float:
    """
    Tính căn bậc 2 của một số.
    :param num: số cần tính căn
    :return: căn bậc 2 của num
    """
    return math.sqrt(float(num))

@tool
def divide(num: float, divisor: float) -> float:
    """
    Chia hai số.
    :param num: số bị chia
    :param divisor: số chia
    :return: kết quả của phép chia num cho divisor
    """
    return float(num) / float(divisor)


tools = [TavilySearch(max_results=1), triple, modulo, power, square_root, divide]

In [4]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o").bind_tools(tools)

### __Agent__

In [5]:
from langchain_core.messages import SystemMessage

system_prompt = SystemMessage(
    content="""
Bạn là một reasoning agent thông minh. Hãy luôn suy nghĩ từng bước một (step-by-step).

Hãy sử dụng định dạng sau:

Thought: suy nghĩ của bạn về việc cần làm tiếp theo (viết bằng Tiếng Việt)
Action: hành động cần thực hiện, ví dụ: `search`, `triple`
Action Input: đầu vào cho hành động đó
Observation: kết quả trả về của hành động

(Lặp lại quá trình Thought/Action/Observation nếu cần thiết)

Final Answer: câu trả lời cuối cùng gửi tới người dùng (viết bằng Tiếng Việt)

Question: {input}
"""
)

In [6]:
import json
from langchain_core.messages import ToolMessage
from langchain_core.runnables import RunnableConfig

tools_by_name = {tool.name: tool for tool in tools}


# Define tool node
def tool_node(state: AgentState):
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=json.dumps(tool_result),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outputs}


# Define the node that calls the model
def call_model(
    state: AgentState,
    config: RunnableConfig,
):
    response = model.invoke([system_prompt] + state["messages"], config)
    # We return a list, because this will get added to the existing list
    return {"messages": [response]}


# Define the conditional edge that determines whether to continue or not
def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    # If there is no function call, then we finish
    if not last_message.tool_calls:
        return "end"
    # Otherwise if there is, we continue
    else:
        return "continue"

### __Graph__

In [7]:
from langgraph.graph import StateGraph, END

# Define a new graph
workflow = StateGraph(AgentState)

# Define the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Set the entrypoint as "agent"
# This means that this node is the first one called
workflow.set_entry_point("agent")

# Add a conditional edge
workflow.add_conditional_edges(
    # First, we define the start node. We use "agent".
    # This means these are the edges taken after the "agent" node is called.
    "agent",
    # Next, we pass in the function that will determine which node is called next.
    should_continue,
    # Finally we pass in a mapping.
    # Based on which one the output matches, that node will then be called.
    {
        # If "tools", then we call the tool node.
        "continue": "tools",
        # Otherwise we finish.
        "end": END,
    },
)

# We now add a normal edge from "tools" to "agent".
# This means that after "tools" is called, "agent" node is called next.
workflow.add_edge("tools", "agent")

# Now we can compile the graph
graph = workflow.compile()

In [8]:
graph.get_graph().draw_mermaid_png(output_file_path="static/react_agent_graph.png")
graph.get_graph().print_ascii()

        +-----------+         
        | __start__ |         
        +-----------+         
              *               
              *               
              *               
          +-------+           
          | agent |           
          +-------+           
         .         .          
       ..           ..        
      .               .       
+-------+         +---------+ 
| tools |         | __end__ | 
+-------+         +---------+ 


### __Test__

In [9]:
inputs = {"messages": [("user", "Tìm dân số của Tokyo và Seoul (tính bằng triệu người). Lấy dân số Tokyo chia lấy dư cho 10, sau đó cộng với căn bậc 2 của dân số Seoul (làm tròn xuống thành số nguyên), cuối cùng nhân kết quả đó với 3. Hãy tính từng bước một cách chính xác.")]}

for state in graph.stream(inputs, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()

================================ Human Message =================================

Tìm dân số của Tokyo và Seoul (tính bằng triệu người). Lấy dân số Tokyo chia lấy dư cho 10, sau đó cộng với căn bậc 2 của dân số Seoul (làm tròn xuống thành số nguyên), cuối cùng nhân kết quả đó với 3. Hãy tính từng bước một cách chính xác.
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_gVGDpCMVWrui8xfvVgvU4Qxj)
 Call ID: call_gVGDpCMVWrui8xfvVgvU4Qxj
  Args:
    query: dân số Tokyo 2023
  tavily_search (call_DAZtnWncebwyfITqTzsnnfAz)
 Call ID: call_DAZtnWncebwyfITqTzsnnfAz
  Args:
    query: dân số Seoul 2023
================================= Tool Message =================================
Name: tavily_search

{"query": "d\u00e2n s\u1ed1 Seoul 2023", "response_time": 0.64, "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://world.kbs.co.kr/service/news_view.htm?lang=v&Seq_Code=63188", "title": "T\u1ed5n